# 1. Library calling

In [1]:
#pip install selenium
#pip install webdriver_manager
from selenium import webdriver
from selenium.webdriver.common.by import By
from time import sleep
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from urllib.parse import unquote
from selenium.webdriver.support.ui import Select
import pandas as pd
import datetime 

import warnings

# Ignore all warnings (not recommended in general)
warnings.filterwarnings("ignore")

# 2. Defining the Product Information and Location

In [2]:
SummaryFolder=r'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects'
summaryFile='Scraping_List.txt'
st=pd.read_csv(SummaryFolder+'\\'+summaryFile)
print(st)
#Define Product to extract
search_text = st['Product Name'][57]
print(search_text)
Source="Amazon"
OFolder=fr'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\{Source}\Outputs'
IFolder=fr'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\{Source}\Inputs'

                           Product Name
0                         Ignition Coil
1               Windshield Washer Pumps
2                 coupler trailer locks
3   Adjustable Trailer Hitch Ball Mount
4                        Vacuum Cleaner
..                                  ...
70               swing away hitch mount
71                      Garage Products
72                        Oxygen Sensor
73             Brand Specific O2 Sensor
74                                  FOB

[75 rows x 1 columns]
Motor Cycle Bags


# 3. Setting Webdriver and Website Specific Information

In [3]:
path= 'C://chromedriver.exe'
driver=webdriver.Chrome()
driver.get('https://www.amazon.in/')

### Please update capatche.

In [4]:
location = driver.find_element(By.ID, 'nav-global-location-popover-link')

location.click()
sleep(1)
pin = driver.find_element(By.ID, 'GLUXZipUpdateInput')
pinnumber='562130'
pin.send_keys(pinnumber)

apply=driver.find_element(By.CSS_SELECTOR, 'span.a-button-inner[data-action="GLUXPostalUpdateAction"]')
apply.click()


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":"[id="nav-global-location-popover-link"]"}
  (Session info: chrome=145.0.7632.76); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff689404435
	0x7ff689404490
	0x7ff68919d49d
	0x7ff6891f82fe
	0x7ff6891f860c
	0x7ff689248b67
	0x7ff689245728
	0x7ff6891e8e38
	0x7ff6891e9d23
	0x7ff6896d3b30
	0x7ff6896ce3b8
	0x7ff6896ee72a
	0x7ff6894207e5
	0x7ff68942978c
	0x7ff68940d794
	0x7ff68940d946
	0x7ff6893f3457
	0x7ffe4b917374
	0x7ffe4d13cc91


In [5]:
#driver.maximize_window()

In [7]:
dropdown = driver.find_element(By.ID,'searchDropdownBox')

# Create a Select object
select = Select(dropdown)

# Select an option by visible text
select.select_by_visible_text('Automotive Parts & Accessories')

In [10]:
# Find the search bar by its ID
try:
    search_bar = driver.find_element(By.ID, 'nav-bb-search')
except:
    search_bar=driver.find_element(By.ID, "twotabsearchtextbox")

# Enter the search text

search_bar.send_keys(search_text)

# Submit the form to perform the search
search_bar.submit()

# 4. Defining the Dataframe and Extracting the data into the Dataframe

In [17]:
Cols=["Links","Name"]
df = pd.DataFrame(columns=Cols)
count=0

In [6]:
Linklist=driver.find_elements(By.CSS_SELECTOR,'[class="a-link-normal aok-block"]')
Namelist=driver.find_elements(By.CSS_SELECTOR,'[class="_cDEzb_p13n-sc-css-line-clamp-3_g3dy1"]')

for link,name in zip(Linklist,Namelist):
    df.loc[count,"Links"]=link.get_attribute("href").split("/ref")[0]
    df.loc[count,"Name"]=name.text
    count=count+1

In [7]:
df

,Links,Name


In [18]:
for i in range(5):
    elemento = driver.find_element(By.CLASS_NAME,"a-spacing-small")
    # Print the text content of the element
    #print("Element Text:", element.text)
    PageNumber=elemento.text.split('result')[0]
    PageRange= PageNumber.split(" ")[0]
    TotalPage=PageNumber.split(" ")[2]
    max= PageRange.split("-")[1]
    print(max, TotalPage)
    sleep(5)
    elements=driver.find_elements(By.CLASS_NAME,'s-title-instructions-style')
    for element in elements:
        try:
            df.loc[count,'Brand']=element.find_element(By.CSS_SELECTOR,'[class="a-size-base-plus a-color-base"]').text
        except:
            pass
        df.loc[count,'Name']=element.find_element(By.CLASS_NAME,"a-text-normal").text
        df.loc[count,'Links']=element.find_element(By.CLASS_NAME,"a-text-normal").get_attribute('href')
        try:
            df.loc[count,'Sponsored?']=element.find_element(By.CSS_SELECTOR,'[class="a-color-secondary"]').text
        except:
            pass
        count=count+1
    if max!= TotalPage:
        next_button = driver.find_element(By.CLASS_NAME,'s-pagination-next')
        next_button.click()
        i=i+1
    else:
        break

24 514
48 514
48 514
72 514
96 514


In [20]:
df

,Links,Name,Sponsored?
0,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...,LG 126 cm (50 inches) UA82 Series 4K Ultra HD ...,Sponsored
1,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...,Wobble 138.7 cm (55 inches) K Series 4K UHD Sm...,Sponsored
2,https://www.amazon.in/Visio-World-inches-VW32A...,VW 80 cm (32 inches) Frameless Series HD Ready...,NaN
3,https://www.amazon.in/Samsung-inches-Smart-LED...,Samsung 80 cm (32 inches) HD Smart LED TV UA32...,NaN
4,https://www.amazon.in/Visio-World-inches-VW32S...,VW 80 cm (32 inches) Frameless Series HD Ready...,NaN
...,...,...,...
192,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...,Xiaomi 138 cm (55 inch) FX Ultra HD 4K Smart L...,Sponsored
193,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...,LG 108 cm (43 inches) UA82 Series 4K Ultra HD ...,NaN
194,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...,coocaa Frameless 164 cm (65 inch) Frameless QL...,NaN
195,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...,Philips 138 cm (55 inch) QLED MINI LED Smart F...,NaN


# 5. Post Processing Data and Exporting

In [21]:
items_to_remove = ["javascript:void(0)"]
df = df[~df['Links'].isin(items_to_remove)]
df

,Links,Name,Sponsored?
0,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...,LG 126 cm (50 inches) UA82 Series 4K Ultra HD ...,Sponsored
1,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...,Wobble 138.7 cm (55 inches) K Series 4K UHD Sm...,Sponsored
2,https://www.amazon.in/Visio-World-inches-VW32A...,VW 80 cm (32 inches) Frameless Series HD Ready...,NaN
3,https://www.amazon.in/Samsung-inches-Smart-LED...,Samsung 80 cm (32 inches) HD Smart LED TV UA32...,NaN
4,https://www.amazon.in/Visio-World-inches-VW32S...,VW 80 cm (32 inches) Frameless Series HD Ready...,NaN
...,...,...,...
192,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...,Xiaomi 138 cm (55 inch) FX Ultra HD 4K Smart L...,Sponsored
193,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...,LG 108 cm (43 inches) UA82 Series 4K Ultra HD ...,NaN
194,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...,coocaa Frameless 164 cm (65 inch) Frameless QL...,NaN
195,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...,Philips 138 cm (55 inch) QLED MINI LED Smart F...,NaN


In [22]:
df['Links1']=df['Links'].str.replace('%',"&").str.replace('&2F','/')

In [23]:
print(len(df))
df

197


,Links,Name,Sponsored?,Links1
0,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...,LG 126 cm (50 inches) UA82 Series 4K Ultra HD ...,Sponsored,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...
1,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...,Wobble 138.7 cm (55 inches) K Series 4K UHD Sm...,Sponsored,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...
2,https://www.amazon.in/Visio-World-inches-VW32A...,VW 80 cm (32 inches) Frameless Series HD Ready...,NaN,https://www.amazon.in/Visio-World-inches-VW32A...
3,https://www.amazon.in/Samsung-inches-Smart-LED...,Samsung 80 cm (32 inches) HD Smart LED TV UA32...,NaN,https://www.amazon.in/Samsung-inches-Smart-LED...
4,https://www.amazon.in/Visio-World-inches-VW32S...,VW 80 cm (32 inches) Frameless Series HD Ready...,NaN,https://www.amazon.in/Visio-World-inches-VW32S...
...,...,...,...,...
192,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...,Xiaomi 138 cm (55 inch) FX Ultra HD 4K Smart L...,Sponsored,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...
193,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...,LG 108 cm (43 inches) UA82 Series 4K Ultra HD ...,NaN,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...
194,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...,coocaa Frameless 164 cm (65 inch) Frameless QL...,NaN,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...
195,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...,Philips 138 cm (55 inch) QLED MINI LED Smart F...,NaN,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...


In [24]:
df['ASINs'] = df['Links1'].str.split('''/dp/''',expand=True)[1].str.split('''/ref''',expand=True)[0]
del df["Links1"]
prefix="https://www.amazon.in/dp/"
df['Links']=prefix+df["ASINs"]


In [25]:
df=df.drop_duplicates(subset=['ASINs'])
df

,Links,Name,Sponsored?,ASINs
0,https://www.amazon.in/dp/B0FB35J4B6,LG 126 cm (50 inches) UA82 Series 4K Ultra HD ...,Sponsored,B0FB35J4B6
1,https://www.amazon.in/dp/B0FP2TB2C4,Wobble 138.7 cm (55 inches) K Series 4K UHD Sm...,Sponsored,B0FP2TB2C4
2,https://www.amazon.in/dp/B07MKFNHKG,VW 80 cm (32 inches) Frameless Series HD Ready...,NaN,B07MKFNHKG
3,https://www.amazon.in/dp/B0F84FBWQM,Samsung 80 cm (32 inches) HD Smart LED TV UA32...,NaN,B0F84FBWQM
4,https://www.amazon.in/dp/B07MNNH484,VW 80 cm (32 inches) Frameless Series HD Ready...,NaN,B07MNNH484
...,...,...,...,...
187,https://www.amazon.in/dp/B0DKNLRY8Q,acer 126 cm (50 inches) G Plus Series 4K Ultra...,NaN,B0DKNLRY8Q
188,https://www.amazon.in/dp/B0B9XT82V2,acer 139 cm (55 inches) W Series 4K Ultra HD Q...,NaN,B0B9XT82V2
189,https://www.amazon.in/dp/B0DZHMX6D3,TCL 80 cms (32 inches) V4C Series HD Ready Sma...,NaN,B0DZHMX6D3
190,https://www.amazon.in/dp/B0DW1C7X4C,VW 165 cm (65 inches) Pro Series 4K Ultra HD S...,NaN,B0DW1C7X4C


In [26]:
search_text="Television"

In [27]:
df.to_excel(IFolder+'\\'+f'{Source}_IN_ProductLinks_'+search_text+'.xlsx', index=False)#

# 99. Archived Codes

In [ ]:
sortbutton=driver.find_element(By.XPATH,"//span[@id='a-autoid-0-announce']")
#sortbutton=driver.find_element(By.CLASS_NAME,"a-button-text a-declarative")
sortbutton.click()
BSbutton=driver.find_element(By.XPATH,"//a[contains(@class, 'a-dropdown-link') and text()='Best Sellers']")
BSbutton.click()